<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session11-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 11: in-class discussion problem (3 of 3)

This is the block 3 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then run the code below and look at what it does. There is one mistake in it, but finding that is
only the way in: the point is to work out why the result comes out as it does. Discuss that with
your neighbour, then answer the quiz again.

## Train on eukaryotes, test on bacteria

SignalP 5.0 needed to be told the organism group of every sequence. SignalP 6.0 does not: its
authors report that the protein language model infers the taxonomic context from the sequence
itself. Bacterial and eukaryotic signal peptides share the n/h/c-region plan but differ in details:
bacterial ones tend to be longer, with more positively charged n-regions.

We test the idea on our data. Two signal-peptide detectors are trained on **eukaryotic** proteins
only and tested on **bacterial** proteins (and the other way round): a logistic regression on the
one-hot encoded first 70 residues, and one on frozen ESM-2 embeddings (`esm2_t12_35M`, averaged over
the same 70 residues; computed in advance, since that takes about five minutes on a CPU). How well
does each keep its performance in the other kingdom?

**Fix and run.** Run both cells once as they are. The line marked `FIX` decides which proteins each
model is tested on, and it picks the wrong ones. Fix it and run the second cell again.

In [1]:
import io, os, requests, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import matthews_corrcoef

REPO = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/"
def course_file(name):
    local = os.path.join("data", name)
    return open(local, "rb").read() if os.path.exists(local) else requests.get(REPO + name, timeout=60).content

table = pd.read_csv(io.BytesIO(course_file("session11-signal-peptides.tsv")), sep="\t")
y = table.sp_end.notna().to_numpy().astype(int)
eukaryote = table.group.str.startswith("euk").to_numpy()
print(f"eukaryotes: {eukaryote.sum()} proteins ({y[eukaryote].sum()} with a signal peptide); "
      f"bacteria: {(~eukaryote).sum()} ({y[~eukaryote].sum()} with one)")

N, AA = 70, "ACDEFGHIKLMNPQRSTVWY"
onehot = np.zeros((len(table), N * 20), dtype=np.float32)
for n, s in enumerate(table.sequence):
    for p, ch in enumerate(s[:N]):
        onehot[n, p * 20 + AA.index(ch)] = 1.0

stored = np.load(io.BytesIO(course_file("session11-signal-peptides-esm2-35M.npz")))
assert (stored["accession"] == table.accession.to_numpy()).all()
emb = stored["embedding"].astype(np.float32)
print("features: one-hot", onehot.shape, " ESM-2", emb.shape)

eukaryotes: 2645 proteins (1158 with a signal peptide); bacteria: 2265 (778 with one)
features: one-hot (4910, 1400)  ESM-2 (4910, 480)


In [2]:
print(f"{'features':10s} {'train euk, test bac':>22s} {'train bac, test euk':>22s}")
for name, F in [("one-hot", onehot), ("ESM-2", emb)]:
    mcc = []
    for train, test in [(eukaryote, ~eukaryote), (~eukaryote, eukaryote)]:
        model = LogisticRegression(max_iter=3000).fit(F[train], y[train])
        tested = train                     # FIX: test on the other kingdom
        mcc.append(matthews_corrcoef(y[tested], model.predict(F[tested])))
    print(f"{name:10s} {mcc[0]:22.3f} {mcc[1]:22.3f}")

features      train euk, test bac    train bac, test euk


one-hot                     1.000                  1.000


ESM-2                       0.961                  0.977


**If you finish early.**

1. Why might transfer from bacteria to eukaryotes be harder than the other way round? Look at the
   number of training proteins in each direction.
2. SignalP 6.0 no longer asks for the organism group. What more would you want to see before
   concluding that organism information is unnecessary? Think of archaea (absent from our data),
   other signal-peptide types, and a single train/test split.

<details>
<summary><b>Discussion point</b> &mdash; open this once you have run the code and talked it through</summary>

As it was, each model was tested on the proteins it had been trained on,
and both looked almost perfect: one-hot 1.000 in both directions, ESM-2 0.96 and 0.98. On training
data, the one-hot model even looks better. The fix is `tested = test`:

| features | train euk, test bac | train bac, test euk |
|---|---|---|
| one-hot | 0.647 | 0.478 |
| ESM-2 | 0.953 | 0.787 |

The one-hot model had memorised its training proteins and loses much of its performance in the other
kingdom; ESM-2 keeps most of it, because its representation was learnt from millions of unlabelled
sequences from all kingdoms. Transfer from bacteria to eukaryotes is harder for both, with fewer
training proteins (2,265 against 2,645). This agrees with SignalP 6.0's claim, but one split on two
kingdoms with one signal-peptide type is not yet proof that organism information is unnecessary.

</details>